# 14. 다른 프로그램과 대화하기 — HTTP와 API 호출

| 구성 | 의미 | LLM API 호출에서의 예 |
| --- | --- | --- |
| **메서드** | 무엇을 하려는가 | `POST` (대화를 보내고 답을 받는다) |
| **주소(URL)** | 어디에 요청하는가 | `https://api.openai.com/v1/chat/completions` |
| **헤더(header)** | 요청에 대한 부가 정보 | 인증 키(`Authorization`), 본문 형식(`Content-Type`) |
| **본문(body)** | 보내는 데이터 | 모델 이름과 메시지 목록 (JSON) |

In [7]:
import requests

url = "https://jsonplaceholder.typicode.com/posts/1"
response = requests.get(url, timeout=10)

print(response.status_code)
print(response.headers["Content-Type"])

data = response.json()
print(type(data))
print(data["id"], data["userId"])
print(data["title"])

200
application/json; charset=utf-8
<class 'dict'>
1 1
sunt aut facere repellat provident occaecati excepturi optio reprehenderit


In [8]:
data

{'userId': 1,
 'id': 1,
 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit',
 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}

In [9]:
data["body"]

'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'

In [10]:
response.headers

{'Date': 'Fri, 27 Nov 2026 03:14:43 GMT', 'Content-Type': 'application/json; charset=utf-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'access-control-allow-credentials': 'true', 'Cache-Control': 'max-age=43200', 'etag': 'W/"124-yiKdLzqO5gfBrJFrcdJ8Yq0LGnU"', 'expires': '-1', 'pragma': 'no-cache', 'vary': 'Origin, Accept-Encoding', 'x-content-type-options': 'nosniff', 'x-powered-by': 'Express', 'x-ratelimit-limit': '3600000', 'x-ratelimit-remaining': '3599992', 'x-ratelimit-reset': '1790812826', 'Age': '10429', 'Nel': '{"report_to":"cf-nel","success_fraction":0.0,"max_age":604800}', 'cf-cache-status': 'HIT', 'Report-To': '{"group":"cf-nel","max_age":604800,"endpoints":[{"url":"https://a.nel.cloudflare.com/report/v4?s=fEMyKwhhPH4ZlnR7YX7uWQQNiN28IftLUIvH1lIxDXPoLhEVgqqDICs42p4%2BfUQkiNtVG0yp1NZVY0dnHH%2FnKhWTeFpj%2BBj0VLfC9poPFlUeWqpxg2HYJm51nlcj9wBpKANvOOBbtdAE1bN3Ol5A"}]}', 'Content-Encoding': 'gzip', 'Server': 'cloudflare', 'CF-RAY': 'a60df6dce8a6061e-LAX', 'alt-svc

In [12]:
response = requests.get(
    "https://jsonplaceholder.typicode.com/comments",
    params={"postId": 1},
    timeout=10
)
print(response.url)

comments = response.json()
print(type(comments), len(comments))
print(comments[0]["email"])

https://jsonplaceholder.typicode.com/comments?postId=1
<class 'list'> 5
Eliseo@gardner.biz


In [16]:
new_post = {
    "title": "연차 신청 질문",
    "body": "연차는 며칠 전까지 신청해야 하나요?",
    "userId": 1,
}
response = requests.post(
    "https://jsonplaceholder.typicode.com/posts",
    json=new_post,
    timeout=10,
)
print(response.status_code)
print(response.json())

201
{'title': '연차 신청 질문', 'body': '연차는 며칠 전까지 신청해야 하나요?', 'userId': 1, 'id': 101}


In [18]:
# 예시
import os
import requests

response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers={"Authorization": f"Bearer {os.getenv('OPENAI_API_KEY')}"},
    json={
        "model": "gpt-6-luna",
        "messages": [
            {"role": "system", "content": "너는 사내 규정 도우미다."},
            {"role": "user", "content": "연차는 며칠이야?"},
        ],
    },
    timeout=30,
)
reply = response.json()["choices"][0]["message"]["content"]

KeyError: 'choices'

In [19]:
import requests


def fetch_post(post_id: int) -> dict | None:
    url = f"https://jsonplaceholder.typicode.com/posts/{post_id}"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()             # 4xx, 5xx이면 예외를 일으킨다
        return response.json()
    except requests.HTTPError as e:
        print(f"HTTP 에러: {e}")
    except requests.Timeout:
        print("응답 시간 초과")
    except requests.ConnectionError:
        print("서버에 연결할 수 없다")
    return None


print(fetch_post(1)["id"])
print(fetch_post(999))

1
HTTP 에러: 404 Client Error: Not Found for url: https://jsonplaceholder.typicode.com/posts/999
None
